**This Notebook is responsible for transforming raw data to a silver data.** 

In [1]:
df = spark.table('FraudDetection.dbo.Bronze')

StatementMeta(, cf5464a4-2db9-4071-9e1e-7ad7e349d349, 3, Finished, Available, Finished, False)

In [2]:
df.count()

StatementMeta(, cf5464a4-2db9-4071-9e1e-7ad7e349d349, 4, Finished, Available, Finished, False)

21000000

In [10]:
from pyspark.sql import functions as F

# ===============================
# 0. Read the table from bronze layer
# ===============================
df = spark.sql("SELECT * FROM FraudDetection.dbo.Bronze")


# =========================
# 1. Separate label
# =========================
y_col = "isFraud"

# Drop leakage columns
df = df.drop("isFlaggedFraud")

# =========================
# 2. Balance features (VERY IMPORTANT)
# =========================
df = df.withColumn(
    "orig_balance_diff",
    F.col("oldbalanceOrg") - F.col("newbalanceOrig") - F.col("amount")
)

df = df.withColumn(
    "dest_balance_diff",
    F.col("newbalanceDest") - F.col("oldbalanceDest") - F.col("amount")
)

# =========================
# 3. Zero balance indicators
# =========================
df = df.withColumn(
    "orig_zero",
    (F.col("oldbalanceOrg") == 0).cast("int")
)

df = df.withColumn(
    "dest_zero",
    (F.col("oldbalanceDest") == 0).cast("int")
)

# =========================
# 4. Amount features
# =========================
df = df.withColumn(
    "log_amount",
    F.log1p(F.col("amount"))
)

df = df.withColumn(
    "amount_to_balance",
    F.col("amount") / (F.col("oldbalanceOrg") + F.lit(1))
)

# =========================
# 5. Error flags (VERY POWERFUL)
# =========================
df = df.withColumn(
    "orig_error",
    ((F.col("oldbalanceOrg") - F.col("amount")) != F.col("newbalanceOrig")).cast("int")
)

df = df.withColumn(
    "dest_error",
    ((F.col("oldbalanceDest") + F.col("amount")) != F.col("newbalanceDest")).cast("int")
)

# =========================
# 6. Extract entity type from names
# =========================
df = df.withColumn(
    "orig_name_type",
    F.upper(F.substring(F.col("nameOrig"), 1, 1))
)

df = df.withColumn(
    "dest_name_type",
    F.upper(F.substring(F.col("nameDest").cast('string'), 1, 1))
)


# =========================
# 7. Frequency features (IMPORTANT)
# =========================
orig_counts = df.groupBy("nameOrig").count().withColumnRenamed("count", "orig_name_freq")
dest_counts = df.groupBy("nameDest").count().withColumnRenamed("count", "dest_name_freq")

df = df.join(orig_counts, on="nameOrig", how="left")
df = df.join(dest_counts, on="nameDest", how="left")


# =========================
# 8. Pair frequency (VERY STRONG FEATURE)
# =========================
pair_counts = df.groupBy("nameOrig", "nameDest").count().withColumnRenamed("count", "pair_freq")

df = df.join(pair_counts, on=["nameOrig", "nameDest"], how="left")


# -------------------------
# 9. Balance ratio features
# -------------------------
df = df.withColumn(
    "balance_ratio_orig",
    F.col("newbalanceOrig") / (F.col("oldbalanceOrg") + 1)
)

df = df.withColumn(
    "balance_ratio_dest",
    F.col("newbalanceDest") / (F.col("oldbalanceDest") + 1)
)

# -------------------------
# 10. Absolute differences
# -------------------------
df = df.withColumn("abs_orig_diff", F.abs(F.col("orig_balance_diff")))
df = df.withColumn("abs_dest_diff", F.abs(F.col("dest_balance_diff")))

# -------------------------
# 11. Amount anomaly features
# -------------------------
df = df.withColumn(
    "amount_vs_orig",
    F.col("amount") / (F.col("oldbalanceOrg") + 1)
)

df = df.withColumn(
    "amount_vs_dest",
    F.col("amount") / (F.col("oldbalanceDest") + 1)
)

# -------------------------
# 12. Zero behavior flags
# -------------------------
df = df.withColumn(
    "orig_zero_flag",
    F.when(
        (F.col("oldbalanceOrg") == 0) & (F.col("newbalanceOrig") == 0),
        1
    ).otherwise(0)
)

df = df.withColumn(
    "dest_zero_flag",
    F.when(
        (F.col("oldbalanceDest") == 0) & (F.col("newbalanceDest") == 0),
        1
    ).otherwise(0)
) 

# -------------------------
# 14. Time features
# -------------------------
df = df.withColumn("hour", F.col("step") % 24)
df = df.withColumn(
    "is_night",
    F.when(F.col("hour").isin([0,1,2,3,4]), 1).otherwise(0)
)

# -------------------------
# 15. Non-linear transforms
# -------------------------
df = df.withColumn("sqrt_amount", F.sqrt(F.col("amount")))
df = df.withColumn("log_balance_orig", F.log1p(F.col("oldbalanceOrg")))
df = df.withColumn("log_balance_dest", F.log1p(F.col("oldbalanceDest")))




# =========================
# Display the final result
# =========================
display(df.limit(10))



StatementMeta(, 818c8cea-967f-449c-b312-04229f4a723e, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 420e3052-f4ac-465a-a2da-9b3df5ce3894)

In [11]:
df_ = df.drop('nameOrig')

StatementMeta(, 818c8cea-967f-449c-b312-04229f4a723e, 13, Finished, Available, Finished, False)

In [12]:
df_finale = df_.drop('nameDest')

StatementMeta(, 818c8cea-967f-449c-b312-04229f4a723e, 14, Finished, Available, Finished, False)

In [15]:
display(df_finale.limit(10))
df_finale.write.mode('overwrite').format('delta').saveAsTable('FraudDetection.dbo.silver_layer_ciferFraudData_2')

StatementMeta(, 818c8cea-967f-449c-b312-04229f4a723e, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 15e8c5e0-21f9-48a0-b946-47d5ce20aca4)

In [1]:
# Read the delta table 
spark_df = spark.read.table('FraudDetection.dbo.silver_layer_ciferfrauddata_2')#.select(features)



StatementMeta(, 2f86f3cc-d85c-437d-a32c-aeb33c380d4b, 3, Finished, Available, Finished, False)

In [19]:
spark_df.count()

StatementMeta(, 818c8cea-967f-449c-b312-04229f4a723e, 21, Finished, Available, Finished, False)

21000000

In [3]:
# Down sampling nonfraud records: Note that this is only used for model training in this learning process
fraud_df = spark_df.filter("isFraud = 1")
non_fraud_df = spark_df.filter("isFraud = 0")

on_fraud_sample = non_fraud_df.sample(fraction=0.0071)


balanced_df = on_fraud_sample.union(fraud_df)

balanced_df.groupBy('isFraud').count().show()



StatementMeta(, 2f86f3cc-d85c-437d-a32c-aeb33c380d4b, 5, Finished, Available, Finished, False)

[('step', 'bigint'), ('type', 'string'), ('amount', 'double'), ('oldbalanceOrg', 'double'), ('newbalanceOrig', 'double'), ('oldbalanceDest', 'double'), ('newbalanceDest', 'double'), ('isFraud', 'bigint'), ('orig_balance_diff', 'double'), ('dest_balance_diff', 'double'), ('orig_zero', 'int'), ('dest_zero', 'int'), ('log_amount', 'double'), ('amount_to_balance', 'double'), ('orig_error', 'int'), ('dest_error', 'int'), ('orig_name_type', 'string'), ('dest_name_type', 'string'), ('orig_name_freq', 'bigint'), ('dest_name_freq', 'bigint'), ('pair_freq', 'bigint'), ('balance_ratio_orig', 'double'), ('balance_ratio_dest', 'double'), ('abs_orig_diff', 'double'), ('abs_dest_diff', 'double'), ('amount_vs_orig', 'double'), ('amount_vs_dest', 'double'), ('orig_zero_flag', 'int'), ('dest_zero_flag', 'int'), ('hour', 'bigint'), ('is_night', 'int'), ('sqrt_amount', 'double'), ('log_balance_orig', 'double'), ('log_balance_dest', 'double')]
+-------+------+
|isFraud| count|
+-------+------+
|      0|148

In [8]:
print(balanced_df.columns)

StatementMeta(, 2f86f3cc-d85c-437d-a32c-aeb33c380d4b, 10, Finished, Available, Finished, False)

['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'orig_balance_diff', 'dest_balance_diff', 'orig_zero', 'dest_zero', 'log_amount', 'amount_to_balance', 'orig_error', 'dest_error', 'orig_name_type', 'dest_name_type', 'orig_name_freq', 'dest_name_freq', 'pair_freq', 'balance_ratio_orig', 'balance_ratio_dest', 'abs_orig_diff', 'abs_dest_diff', 'amount_vs_orig', 'amount_vs_dest', 'orig_zero_flag', 'dest_zero_flag', 'hour', 'is_night', 'sqrt_amount', 'log_balance_orig', 'log_balance_dest']


In [5]:
balanced_df.write.mode('overwrite').format('delta').saveAsTable('FraudDetection.dbo.silver_data_for_MLmodel')

StatementMeta(, d64526c5-8a58-4bbb-bd65-31ae56a1bde5, 7, Finished, Available, Finished, False)

In [4]:
%%sql
select 
    *
from FraudDetection.dbo.silver_data_for_MLmodel

StatementMeta(, 587e7f19-d88b-4e8d-8227-01bb3f161240, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 21 fields>

In [5]:
df = spark.sql("SELECT * FROM FraudDetection.dbo.silver_data_for_mlmodel where isFraud = 1")
display(df)

StatementMeta(, c6958385-c86b-40b4-8b16-e9c3fd05a96d, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 71ad289c-079e-46ee-9bbb-5049dbabfb34)

In [13]:
balanced_df.groupBy('isFraud').count().show()

StatementMeta(, 01646891-8054-474c-8aa1-ba2688c8b94e, 15, Finished, Available, Finished, False)

+-------+------+
|isFraud| count|
+-------+------+
|      0|105322|
|      1| 27470|
+-------+------+



In [5]:
df.select('orig_name_type').distinct().show()
df.select('dest_name_type').distinct().show()

StatementMeta(, cf5464a4-2db9-4071-9e1e-7ad7e349d349, 7, Finished, Available, Finished, False)

+--------------+
|orig_name_type|
+--------------+
|             C|
|             S|
+--------------+

+--------------+
|dest_name_type|
+--------------+
|             M|
|             C|
|             S|
+--------------+



In [7]:
# Distribution of the legit and fraudulet transactions 

#count_fraud = spark.table("CiferFraudDetection")

df.groupBy('isFraud').count().show()

# -> Imbalanced dataset : under sampling may help

StatementMeta(, cf5464a4-2db9-4071-9e1e-7ad7e349d349, 9, Finished, Available, Finished, False)

+-------+--------+
|isFraud|   count|
+-------+--------+
|      0|20972530|
|      1|   27470|
+-------+--------+



In [4]:
%%sql
DROP TABLE FraudDetection.dbo.silver_data_for_MLmodel

StatementMeta(, d64526c5-8a58-4bbb-bd65-31ae56a1bde5, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>